In [ ]:
#1️ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
#Read the dataset Q1_data.csv using read_csv()

In [ ]:
# Task 1: Write your code here:
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery =pd.read_csv(delivery_path ) # YOUR CODE HERE

print(f"Shape: {df_delivery.shape}")


In [ ]:
#Inspect the first few rows using head()

In [ ]:
# Task 2: Write your code here:
df_delivery.head()

In [ ]:
#Display dataset information using info()

In [ ]:
# Task 3: Write your code here:
df_delivery.info()

In [ ]:
#Show statistical description using describe()

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
#Plot the target distribution (delivery_time)

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Drop the 'Order_ID' column from the data

In [ ]:
df_delivery.drop(columns=['Order_ID'], inplace=True)

In [ ]:
df_delivery.head()

In [ ]:
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :)
print("Missing values:")
print(df_delivery.isnull().sum())

In [ ]:
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
# df['col'].fillna(df['col'].mode()[0], inplace=True)
df_delivery['Weather'].fillna(df_delivery['Weather'].mode()[0], inplace=True)
df_delivery['Traffic_Level'].fillna(df_delivery['Traffic_Level'].mode()[0], inplace=True)
df_delivery['Time_of_Day'].fillna(df_delivery['Time_of_Day'].mode()[0], inplace=True)
df_delivery['Courier_Experience_yrs'].fillna(df_delivery['Courier_Experience_yrs'].mode()[0], inplace=True)
df_delivery['Delivery_Time'].fillna(df_delivery['Delivery_Time'].mode()[0], inplace=True)

In [ ]:
print("Missing values:")
print(df_delivery.isnull().sum())

In [ ]:
#3=Check and remove duplicates if any exist

In [ ]:
duplicates = df_delivery.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
   print("Dropping Duplicates...")
   df_delivery.drop_duplicates(inplace=True)
   print("Duplicates Dropped.")
else:
   print("No Duplicate Samples Found.")


In [ ]:
#4=Encode categorical variables if needed (Bonus if used One Hot Encoding)

In [ ]:
df_delivery.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_delivery['Weather'] = le.fit_transform(df_delivery['Weather'])
df_delivery['Traffic_Level'] = le.fit_transform(df_delivery['Traffic_Level'])
df_delivery['Time_of_Day'] = le.fit_transform(df_delivery['Time_of_Day'])
df_delivery['Vehicle_Type'] = le.fit_transform(df_delivery['Vehicle_Type'])
df_delivery.head()

In [ ]:
#5=Apply feature scaling for all features (Use StandardScaler)

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
numerical_cols=['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df_delivery[numerical_cols])
df_delivery.head()

In [ ]:
# Task 6: Write your code here:#Check for target imbalance and state if it is imbalanced or not
#not needed

In [ ]:
# Task 1: Write your code here:
X = df_delivery.drop("Delivery_Time",axis=1)
y = df_delivery['Delivery_Time']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
#2=Use the correct split: KFold OR StratifiedKFold

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Use previously generated random classification data
X, y = X.copy(), y.copy()

# Show full dataset class distribution
full_ratio = (y.value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  y class percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40)

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)

In [ ]:
# 3=Train Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
#4=Evaluate using MAE (Mean Absolute Error) ONLY

In [ ]:

from sklearn.metrics import mean_absolute_error
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_absolute_error(y_test, y_pred))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

In [ ]:
#Print the averaged score across all folds

In [ ]:
# Task 2,3,4,5: Write your code here:

In [ ]:

# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': numerical_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: